GND Lookup

In [ ]:
"""
GND-Lookup für Bibliothekslog-Klassifikation
=============================================
Getestetes Format (verifiziert durch Diagnose):
  - Struktur:   [ [ {blank nodes...}, {echte Entität}, {/about} ], [...] ]
  - ijson-Pfad: "item.item"
  - Personen:   preferredNameForThePerson, variantNameForThePerson
  - Sachbegriffe: preferredNameForTheSubjectHeading, variantNameForTheSubjectHeading

Voraussetzungen:
    pip install rapidfuzz pandas tqdm ijson

Downloads:
    https://data.dnb.de/opendata/authorities-gnd-person_lds.jsonld.gz
    https://data.dnb.de/opendata/authorities-gnd-sachbegriff_lds.jsonld.gz
"""

import gzip
import unicodedata
import pickle
import re
from pathlib import Path

import ijson
import pandas as pd
from rapidfuzz import fuzz, process
from tqdm import tqdm


# ── Konfiguration — PFADE ANPASSEN ────────────────────────────────────────────

PERSON_DUMP   = r"authorities-gnd-person_lds.jsonld.gz"
SUBJECT_DUMP  = r"authorities-gnd-sachbegriff_lds.jsonld.gz"
INDEX_CACHE   = r"gnd_index.pkl"
LOG_FILE      = r"input.csv"
OUTPUT_FILE   = r"input_labeled.csv"

# Windows-Beispiel:
# PERSON_DUMP = r"C:\Users\name\Downloads\authorities-gnd-person_lds.jsonld.gz"
# LOG_FILE    = r"C:\Users\name\Downloads\input.csv"

FUZZY_THRESHOLD = 88
MAX_FUZZY_CANDS = 50000


# ── GND-Feldnamen (verifiziert) ────────────────────────────────────────────────

GND = "https://d-nb.info/standards/elementset/gnd#"

PERSON_FIELDS = [
    GND + "preferredNameForThePerson",   # z.B. 'Bauer, Johann Gottfried'
    GND + "variantNameForThePerson",     # alle Namensvarianten
]

SUBJECT_FIELDS = [
    GND + "preferredNameForTheSubjectHeading",  # z.B. 'Abakus'
    GND + "variantNameForTheSubjectHeading",    # z.B. ['Abacus', 'Rechenbrett']
]


# ── Normalisierung ─────────────────────────────────────────────────────────────

def nfc(s: str) -> str:
    """NFD (DNB) → NFC, lowercase, getrimmt."""
    return unicodedata.normalize("NFC", s).lower().strip()


def get_values(entity: dict, field: str) -> list:
    """Extrahiert Stringwerte aus RDF-Wertlisten: [{'@value': 'text'}, ...]"""
    raw = entity.get(field, [])
    if isinstance(raw, str):
        return [raw]
    results = []
    for item in raw:
        if isinstance(item, dict):
            v = item.get("@value", "")
            if v:
                results.append(str(v))
        elif isinstance(item, str) and item:
            results.append(item)
    return results


# ── Index aufbauen ─────────────────────────────────────────────────────────────

def parse_dump(path: str, name_fields: list, label: str) -> dict:
    """
    Baut {normierter_name → gnd_id} Dictionary auf.
    Überspringt Blank Nodes (_:node...) und /about-Metadaten-Objekte.
    """
    index = {}
    path = Path(path)

    if not path.exists():
        print(f"⚠️  Nicht gefunden: {path}")
        print(f"   Download: https://data.dnb.de/opendata/{path.name}")
        return index

    print(f"\n📖 Lese {label}-Dump ...")
    print(f"   (Beim ersten Mal ~15–30 Min. — bitte warten)")

    total = 0
    real  = 0

    with gzip.open(path, "rb") as f:
        for obj in ijson.items(f, "item.item"):
            total += 1

            if total % 500_000 == 0:
                print(f"   ... {total:,} Objekte, {real:,} Entitäten, "
                      f"{len(index):,} Indexeinträge")

            gnd_id = obj.get("@id", "")

            # Blank Nodes und /about-Objekte überspringen
            if (not gnd_id
                    or gnd_id.startswith("_:")
                    or gnd_id.endswith("/about")):
                continue

            real += 1

            # Alle Namenswerte aus den relevanten Feldern einsammeln
            names = []
            for field in name_fields:
                names.extend(get_values(obj, field))

            # Index befüllen
            for name in names:
                key = nfc(name)
                if key and len(key) > 1:
                    index[key] = gnd_id

                    # Zusätzlich: Nachnamen-Teil vor dem Komma separat eintragen
                    # "Pappert, Steffen" → auch "pappert" als Schlüssel
                    # Nur wenn Nachname noch nicht im Index — verhindert dass
                    # häufige Namen ("müller", "jung") thematische Queries
                    # fälschlich als Know-Item klassifizieren
                    if "," in key:
                        surname = key.split(",")[0].strip()
                        if len(surname) > 4 and surname not in index:
                            index[surname] = gnd_id

    print(f"   ✅ {real:,} Entitäten → {len(index):,} Indexeinträge")
    return index


def build_or_load_index() -> tuple:
    cache = Path(INDEX_CACHE)

    if cache.exists():
        print(f"✅ Lade Index aus Cache: {INDEX_CACHE}")
        with open(cache, "rb") as f:
            data = pickle.load(f)
        p, s = data["persons"], data["subjects"]
        print(f"   Personen: {len(p):,} · Sachbegriffe: {len(s):,}")
        return p, s

    print("🔨 Baue GND-Index auf (einmalig) ...")
    persons  = parse_dump(PERSON_DUMP,  PERSON_FIELDS,  "Personen")
    subjects = parse_dump(SUBJECT_DUMP, SUBJECT_FIELDS, "Sachbegriffe")

    with open(cache, "wb") as f:
        pickle.dump({"persons": persons, "subjects": subjects}, f)
    print(f"\n💾 Index gespeichert: {INDEX_CACHE}")
    return persons, subjects


# ── GND-Lookup ─────────────────────────────────────────────────────────────────

def gnd_lookup(query: str, persons: dict, subjects: dict) -> dict:
    """
    Drei Lookup-Ebenen:
      1. Exakter Treffer auf ganzen Query-String
      2. Token-Treffer: jedes Token einzeln prüfen
         ("völkerrecht ipsen" → Token "ipsen" trifft auf Person)
      3. Fuzzy-Treffer für Tippfehler (nur wenn 1+2 nichts fanden)
    """
    q = nfc(query)
    tokens = [t for t in q.split() if len(t) > 2]

    result = {
        "person_exact":   False, "subject_exact":  False,
        "person_token":   False, "subject_token":  False,
        "person_fuzzy":   False, "subject_fuzzy":  False,
        "person_score":   0.0,   "subject_score":  0.0,
        "person_gnd_id":  None,  "subject_gnd_id": None,
        "conflict_same_token": False,
    }

    # 1. Exakter Treffer
    if q in persons:
        result["person_exact"]  = True
        result["person_gnd_id"] = persons[q]
    if q in subjects:
        result["subject_exact"]  = True
        result["subject_gnd_id"] = subjects[q]

    # 2. Token-Treffer — welcher Token hat den Treffer ausgelöst?
    person_hit_token  = None
    subject_hit_token = None

    for token in tokens:
        if token in persons and not result["person_gnd_id"]:
            result["person_token"]  = True
            result["person_gnd_id"] = persons[token]
            person_hit_token = token
        if token in subjects and not result["subject_gnd_id"]:
            result["subject_token"]  = True
            result["subject_gnd_id"] = subjects[token]
            subject_hit_token = token

    # Konfliktlösung: selber Token trifft Person UND Sachbegriff
    # → Sachbegriff gewinnt ("marken" ist Sachbegriff, nicht Eigenname)
    # Verschiedene Token → Person gewinnt ("völkerrecht" + "ipsen")
    result["conflict_same_token"] = (
        person_hit_token is not None and
        subject_hit_token is not None and
        person_hit_token == subject_hit_token
    )

    # 3. Fuzzy (nur wenn noch kein Treffer)
    if not result["person_exact"] and not result["person_token"]:
        prefix = q[0] if q else ""
        cands = {k: v for k, v in persons.items() if k.startswith(prefix)}
        if cands:
            hit = process.extractOne(
                q, list(cands.keys())[:MAX_FUZZY_CANDS], scorer=fuzz.ratio)
            if hit and hit[1] >= FUZZY_THRESHOLD:
                result["person_fuzzy"]  = True
                result["person_score"]  = hit[1] / 100
                result["person_gnd_id"] = cands.get(hit[0])

    if not result["subject_exact"] and not result["subject_token"]:
        prefix = q[0] if q else ""
        cands = {k: v for k, v in subjects.items() if k.startswith(prefix)}
        if cands:
            hit = process.extractOne(
                q, list(cands.keys())[:MAX_FUZZY_CANDS], scorer=fuzz.ratio)
            if hit and hit[1] >= FUZZY_THRESHOLD:
                result["subject_fuzzy"]  = True
                result["subject_score"]  = hit[1] / 100
                result["subject_gnd_id"] = cands.get(hit[0])

    return result


# ── Klassifikation ─────────────────────────────────────────────────────────────

def classify(row: pd.Series) -> str:
    q = str(row.get("keyword", ""))
    n = len(q.split())

    # Regelbasiert — hohe Konfidenz
    if re.search(r'\bOR\b', q):                            return "sonderfall"
    if len(q.strip()) <= 2:                                return "rauschen"
    if re.match(r'^\d+$', q.strip()):                     return "know-item"
    if re.search(r'\b97[89]\d{10}\b', q):                 return "know-item"
    if "<<" in q:                                          return "know-item"
    if re.match(r'^[A-ZÄÖÜa-zäöüß\-]+,\s', q):           return "know-item"
    if re.search(r'\b(1[5-9]|20)\d{2}\b', q) and n >= 2: return "know-item"

    # GND-basiert
    hat_p = row.get("person_exact")  or row.get("person_token")  or row.get("person_fuzzy")
    hat_s = row.get("subject_exact") or row.get("subject_token") or row.get("subject_fuzzy")

    if hat_p and not hat_s: return "know-item"
    if hat_s and not hat_p: return "thematisch"
    if hat_p and hat_s:
        # Konflikt: selber Token trifft Person UND Sachbegriff
        # → "marken" trifft beides → Sachbegriff gewinnt → thematisch
        if row.get("conflict_same_token"):
            return "thematisch"
        # Verschiedene Token: "völkerrecht" + "ipsen" → Person gewinnt → know-item
        return "know-item" if (row.get("person_exact") or row.get("person_token")) \
               else "thematisch"

    # Artikel am Anfang
    if re.match(r'^(der|die|das|ein|eine|the|a|an)\s', q, re.IGNORECASE):
        return "know-item" if n >= 4 else "thematisch"

    # Fallback
    return "explorativ" if n == 1 else "unklar"


# ── Hauptprogramm ──────────────────────────────────────────────────────────────

def main():
    persons, subjects = build_or_load_index()

    if not persons and not subjects:
        print("\n⚠️  Index leer — Pfade in der Konfiguration prüfen.")
        return

    print(f"\n📂 Lade Logdaten: {LOG_FILE}")
    df = pd.read_csv(LOG_FILE, sep=";", encoding="utf-8-sig", on_bad_lines="skip")
    df = df.dropna(subset=["keyword"])
    df["keyword"] = df["keyword"].str.strip()
    df = df[df["keyword"].str.len() > 1]
    print(f"   {len(df):,} Zeilen")

    # Lookup nur auf unique Keywords (vermeidet redundante Arbeit)
    unique_kw = df["keyword"].drop_duplicates().tolist()
    print(f"\n🔍 GND-Lookup für {len(unique_kw):,} unique Keywords ...")

    lookup_results = {}
    for kw in tqdm(unique_kw, unit=" Keywords"):
        lookup_results[kw] = gnd_lookup(kw, persons, subjects)

    lookup_df = pd.DataFrame.from_dict(lookup_results, orient="index")
    lookup_df.index.name = "keyword"
    df = df.merge(lookup_df.reset_index(), on="keyword", how="left")

    df["label"] = df.apply(classify, axis=1)
    df.to_csv(OUTPUT_FILE, index=False, sep=";", encoding="utf-8-sig")

    print(f"\n✅ Gespeichert: {OUTPUT_FILE}")

    print("\n=== Klassenverteilung ===")
    print(df["label"].value_counts().to_string())

    print("\n=== GND-Abdeckung ===")
    for feat in ["person_exact", "person_token", "person_fuzzy",
                 "subject_exact", "subject_token", "subject_fuzzy"]:
        if feat in df.columns:
            n = df[feat].sum()
            print(f"  {feat:25s}: {n:5,} ({n/len(df)*100:.1f}%)")

    print("\n=== Beispiele pro Klasse ===")
    for lbl in ["know-item", "thematisch", "explorativ", "unklar", "sonderfall"]:
        ex = df[df["label"] == lbl]["keyword"].drop_duplicates().head(5).tolist()
        if ex:
            print(f"\n{lbl}:")
            for e in ex:
                print(f"  → {e}")


if __name__ == "__main__":
    main()

Training + Eval

In [ ]:
"""
SBERT Query-Klassifikator — mit GND-Features
=============================================
pip install sentence-transformers scikit-learn numpy

Dateien im gleichen Ordner:
    annotationen.csv
    input_labeled.csv
"""

import pickle
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import classification_report

# ── Daten laden ────────────────────────────────────────────────────────────────

ann = pd.read_csv("annotationen.csv", sep=";", encoding="utf-8-sig")
clf_df = pd.read_csv("input_labeled.csv", sep=";", encoding="utf-8-sig",
                     on_bad_lines="skip")

ann = ann[ann["label"].isin(["know-item", "thematisch"])]

# GND-Features aus klassifizierter Datei holen
gnd_cols = ["keyword", "person_exact", "person_token", "person_fuzzy",
            "subject_exact", "subject_token", "subject_fuzzy",
            "person_score", "subject_score"]
gnd = clf_df[gnd_cols].drop_duplicates("keyword")

# Annotationen mit GND-Features mergen
ann = ann.merge(gnd, on="keyword", how="left")

# Fehlende GND-Features mit 0 auffüllen (Queries ohne GND-Treffer)
gnd_feature_cols = ["person_exact", "person_token", "person_fuzzy",
                    "subject_exact", "subject_token", "subject_fuzzy",
                    "person_score", "subject_score"]
ann[gnd_feature_cols] = ann[gnd_feature_cols].fillna(0).astype(float)

print(f"Trainingsdaten: {len(ann)} Samples")
print(ann["label"].value_counts().to_string())

# ── Embeddings ─────────────────────────────────────────────────────────────────

print("\nBerechne Embeddings ...")
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
X_sbert = model.encode(ann["keyword"].tolist(), show_progress_bar=True)

# GND-Features als zusätzliche Spalten anhängen
X_gnd = ann[gnd_feature_cols].values
X = np.hstack([X_sbert, X_gnd])

print(f"Feature-Vektor: {X_sbert.shape[1]} SBERT + {X_gnd.shape[1]} GND = {X.shape[1]} total")

y = ann["label"].tolist()

# ── Train/Test-Split für ehrliche Evaluation ───────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"\nTrain: {len(y_train)} · Test: {len(y_test)}")

# ── Training + Evaluation ──────────────────────────────────────────────────────

clf = LogisticRegression(max_iter=1000, class_weight="balanced")

# CV auf Trainingsdaten
scores = cross_val_score(clf, X_train, y_train, cv=5, scoring="f1_macro")
print(f"\n5-Fold CV Macro F1 (Train): {scores.mean():.3f} (+/- {scores.std():.3f})")

# Finales Training + Evaluation auf Test-Set
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\nClassification Report (Test-Set — nicht gesehen beim Training):")
print(classification_report(y_test, y_pred, digits=3))

# ── Finales Modell auf allen Daten trainieren ──────────────────────────────────

clf.fit(X, y)

with open("classifier.pkl", "wb") as f:
    pickle.dump(clf, f)
print("Modell gespeichert: classifier.pkl")

# ── Vorhersagen auf unklar/explorativ ─────────────────────────────────────────

df = pd.read_csv("input_labeled.csv", sep=";", encoding="utf-8-sig",
                 on_bad_lines="skip")

needs_ml = df["label"].isin(["unklar", "explorativ"])
ml_df = df[needs_ml].drop_duplicates("keyword")

print(f"\nVorhersage für {len(ml_df)} unklare Queries ...")

X_ml_sbert = model.encode(ml_df["keyword"].tolist(), show_progress_bar=True)
X_ml_gnd   = ml_df[gnd_feature_cols].fillna(0).astype(float).values
X_ml       = np.hstack([X_ml_sbert, X_ml_gnd])

preds = dict(zip(ml_df["keyword"], clf.predict(X_ml)))

df["label_final"] = df["label"]
df.loc[needs_ml, "label_final"] = df.loc[needs_ml, "keyword"].map(preds)

df.to_csv("vorhersagen.csv", index=False, sep=";", encoding="utf-8-sig")
print("\nFinale Verteilung:")
print(df["label_final"].value_counts().to_string())
# Analyse: was klassifiziert ML als Know-Item?
needs_ml = df["label"].isin(["unklar", "explorativ"])
ml_df = df[needs_ml].drop_duplicates("keyword").copy()
ml_df[gnd_feature_cols] = ml_df[gnd_feature_cols].fillna(0).astype(float)

X_ml_sbert = model.encode(ml_df["keyword"].tolist(), show_progress_bar=False)
X_ml = np.hstack([X_ml_sbert, ml_df[gnd_feature_cols].values])
preds = clf.predict(X_ml)
ml_df["pred"] = preds

print("\n=== ML auf unklar/explorativ ===")
print(ml_df["pred"].value_counts())
print(f"Verhältnis: {(ml_df['pred']=='know-item').sum()/(ml_df['pred']=='thematisch').sum():.2f}")
print("\nGespeichert: vorhersagen.csv")

Training ohne GND features (F1 <)

In [ ]:
"""
SBERT Query-Klassifikator — nur Embeddings, keine GND-Features
==============================================================
pip install sentence-transformers scikit-learn

Dateien im gleichen Ordner:
    annotationen.csv
"""

import pickle
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import classification_report

# ── Daten laden ────────────────────────────────────────────────────────────────

ann = pd.read_csv("annotationen.csv", sep=";", encoding="utf-8-sig")
ann = ann[ann["label"].isin(["know-item", "thematisch"])]

print(f"Trainingsdaten: {len(ann)} Samples")
print(ann["label"].value_counts().to_string())

# ── Embeddings ─────────────────────────────────────────────────────────────────

print("\nBerechne Embeddings ...")
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
X = model.encode(ann["keyword"].tolist(), show_progress_bar=True)
y = ann["label"].tolist()

print(f"Feature-Vektor: {X.shape[1]} SBERT-Dimensionen (keine GND-Features)")

# ── Train/Test-Split ───────────────────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(f"\nTrain: {len(y_train)} · Test: {len(y_test)}")

# ── Training + Evaluation ──────────────────────────────────────────────────────

clf = LogisticRegression(max_iter=1000, class_weight="balanced")

scores = cross_val_score(clf, X_train, y_train, cv=5, scoring="f1_macro")
print(f"\n5-Fold CV Macro F1: {scores.mean():.3f} (+/- {scores.std():.3f})")

clf.fit(X_train, y_train)
print("\nClassification Report (Test-Set):")
print(classification_report(y_test, clf.predict(X_test), digits=3))

# Finales Modell auf allen Daten
clf.fit(X, y)
with open("classifier_sbert_only.pkl", "wb") as f:
    pickle.dump(clf, f)
print("Modell gespeichert: classifier_sbert_only.pkl")

Inferenz

In [ ]:
"""
Inferenz — neue Logdaten klassifizieren
========================================
Benutzt classifier.pkl

Dateien im gleichen Ordner:
    classifier.pkl        — trainiertes Modell (392 Features)
    gnd_index.pkl         — GND-Index
    new_input.csv         — neue Daten (gleiche Struktur wie input.csv)

Ausgabe:
    new_output.csv
"""

import pickle
import re
import unicodedata
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm


# ── Konfiguration ──────────────────────────────────────────────────────────────

INPUT_FILE  = r"new_input.csv"
OUTPUT_FILE = r"new_output.csv"
MODEL_FILE  = r"classifier.pkl"
GND_FILE    = r"gnd_index.pkl"
SBERT_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"

# Exakt dieselbe Reihenfolge wie beim Training in train_classifier.py
GND_FEATURE_COLS = [
    "person_exact", "person_token", "person_fuzzy",
    "subject_exact", "subject_token", "subject_fuzzy",
    "person_score",  "subject_score",
]


# ── Normalisierung ─────────────────────────────────────────────────────────────

def nfc(s: str) -> str:
    return unicodedata.normalize("NFC", str(s)).lower().strip()


# ── GND-Lookup (ohne Fuzzy — zu langsam bei 100k Queries) ────────────────────

def gnd_lookup_fast(query: str, persons: dict, subjects: dict) -> dict:
    q = nfc(query)
    tokens = [t for t in q.split() if len(t) > 2]

    result = {
        "person_exact":  False, "subject_exact":  False,
        "person_token":  False, "subject_token":  False,
        "person_fuzzy":  False, "subject_fuzzy":  False,   # immer False (kein Fuzzy)
        "person_score":  0.0,   "subject_score":  0.0,
        "conflict_same_token": False,
    }

    if q in persons:  result["person_exact"]  = True
    if q in subjects: result["subject_exact"] = True

    person_hit_token = subject_hit_token = None
    for token in tokens:
        if token in persons and not result["person_exact"] and not result["person_token"]:
            result["person_token"] = True
            person_hit_token = token
        if token in subjects and not result["subject_exact"] and not result["subject_token"]:
            result["subject_token"] = True
            subject_hit_token = token

    result["conflict_same_token"] = (
        person_hit_token is not None and
        subject_hit_token is not None and
        person_hit_token == subject_hit_token
    )
    return result


# ── Regelbasierte Klassifikation + GND ────────────────────────────────────────

def classify(row: pd.Series):
    """Gibt Label zurück wenn Regel greift, sonst None → ML entscheidet."""
    q = str(row.get("keyword", ""))
    n = len(q.split())

    # Strukturregeln (hohe Konfidenz)
    if re.search(r'\bOR\b', q):                            return "sonderfall"
    if len(q.strip()) <= 2:                                return "rauschen"
    if re.match(r'^\d+$', q.strip()):                     return "know-item"
    if re.search(r'\b97[89]\d{10}\b', q):                 return "know-item"
    if "<<" in q:                                          return "know-item"
    if re.match(r'^[A-ZÄÖÜa-zäöüß\-]+,\s', q):           return "know-item"
    if re.search(r'\b(1[5-9]|20)\d{2}\b', q) and n >= 2: return "know-item"

    # GND-basiert
    hat_p = row.get("person_exact") or row.get("person_token")
    hat_s = row.get("subject_exact") or row.get("subject_token")

    if hat_p and not hat_s: return "know-item"
    if hat_s and not hat_p: return "thematisch"
    if hat_p and hat_s:
        if row.get("conflict_same_token"): return "thematisch"
        return "know-item" if (row.get("person_exact") or row.get("person_token")) \
               else "thematisch"

    if re.match(r'^(der|die|das|ein|eine|the|a|an)\s', q, re.IGNORECASE):
        return "know-item" if n >= 4 else "thematisch"

    return None  # → ML


# ── Hauptprogramm ──────────────────────────────────────────────────────────────

def main():
    # Modell und Index laden
    print("📦 Lade Modell und GND-Index ...")
    with open(MODEL_FILE, "rb") as f:
        clf = pickle.load(f)
    with open(GND_FILE, "rb") as f:
        gnd_data = pickle.load(f)
    persons  = gnd_data["persons"]
    subjects = gnd_data["subjects"]
    sbert    = SentenceTransformer(SBERT_MODEL)
    print(f"   GND: {len(persons):,} Personen · {len(subjects):,} Sachbegriffe")
    print(f"   Modell erwartet: {clf.n_features_in_} Features "
          f"({clf.n_features_in_ - len(GND_FEATURE_COLS)} SBERT + {len(GND_FEATURE_COLS)} GND)")

    # Daten laden
    print(f"\n📂 Lade {INPUT_FILE} ...")
    df = pd.read_csv(INPUT_FILE, sep=";", encoding="utf-8-sig", on_bad_lines="skip")
    df = df.dropna(subset=["keyword"])
    df["keyword"] = df["keyword"].str.strip()
    df = df[df["keyword"].str.len() > 1]
    print(f"   {len(df):,} Zeilen · {df['keyword'].nunique():,} unique Keywords")

    # GND-Lookup auf unique Keywords
    unique_kw = df["keyword"].drop_duplicates().tolist()
    print(f"\n🔍 GND-Lookup für {len(unique_kw):,} unique Keywords ...")
    lookup = {kw: gnd_lookup_fast(kw, persons, subjects) for kw in tqdm(unique_kw)}
    lookup_df = pd.DataFrame.from_dict(lookup, orient="index")
    lookup_df.index.name = "keyword"
    df = df.merge(lookup_df.reset_index(), on="keyword", how="left")

    # Regelbasiert + GND klassifizieren
    df["label"] = df.apply(classify, axis=1)

    # ML auf verbleibende None-Werte (unklar/nicht entschieden)
    needs_ml  = df["label"].isna()
    ml_unique = df[needs_ml]["keyword"].drop_duplicates().tolist()

    print(f"\n   Regelbasiert + GND entschieden: {(~needs_ml).sum():,}")
    print(f"   Verbleibend für ML:              {needs_ml.sum():,} "
          f"({len(ml_unique):,} unique)")

    if ml_unique:
        print("\n🤖 SBERT-Encoding + ML-Klassifikation ...")

        # SBERT-Embeddings
        X_sbert = sbert.encode(ml_unique, show_progress_bar=True, batch_size=64)

        # GND-Features für die ML-Queries — exakt dieselbe Reihenfolge wie beim Training
        ml_gnd = (
            df[df["keyword"].isin(ml_unique)]
            .drop_duplicates("keyword")
            .set_index("keyword")
            .reindex(ml_unique)
        )
        X_gnd = ml_gnd[GND_FEATURE_COLS].fillna(0).astype(float).values

        # Feature-Vektor: 384 SBERT + 8 GND = 392
        X_ml = np.hstack([X_sbert, X_gnd])

        preds = dict(zip(ml_unique, clf.predict(X_ml)))
        df.loc[needs_ml, "label"] = df.loc[needs_ml, "keyword"].map(preds)

    # Speichern
    df.to_csv(OUTPUT_FILE, index=False, sep=";", encoding="utf-8-sig")

    print(f"\n✅ Gespeichert: {OUTPUT_FILE}")
    print("\n=== Klassenverteilung ===")
    print(df["label"].value_counts().to_string())

    ki = (df["label"] == "know-item").sum()
    th = (df["label"] == "thematisch").sum()
    if th > 0:
        print(f"\nVerhältnis KI/TH: {ki/th:.2f}  (Referenz: ~1.42)")


if __name__ == "__main__":
    main()

In [ ]:
"""
Inferenz — neue Logdaten klassifizieren (nur SBERT)
====================================================
Kein GND-Lookup nötig — nur classifier_sbert_only.pkl.

Dateien im gleichen Ordner:
    classifier_sbert_only.pkl
    new_input.csv

Ausgabe:
    new_output.csv
"""

import pickle
import re
import unicodedata
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# ── Konfiguration ──────────────────────────────────────────────────────────────

INPUT_FILE  = r"new_input.csv"
OUTPUT_FILE = r"new_output.csv"
MODEL_FILE  = r"classifier_sbert_only.pkl"
GND_FILE    = r"gnd_index.pkl"
SBERT_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
# Fuzzy-Matching deaktiviert — bringt nur 0.6% Abdeckung aber kostet Stunden
USE_FUZZY   = False


# ── Regelbasierte Vorsortierung ────────────────────────────────────────────────
# Hochkonfidente Fälle direkt entscheiden — kein ML nötig

def classify_rules(q: str) -> str | None:
    """Gibt Label zurück wenn Regel greift, sonst None → ML entscheidet."""
    if re.search(r'\bOR\b', q):                            return "sonderfall"
    if len(q.strip()) <= 2:                                return "rauschen"
    if re.match(r'^\d+$', q.strip()):                     return "know-item"
    if re.search(r'\b97[89]\d{10}\b', q):                 return "know-item"
    if "<<" in q:                                          return "know-item"
    if re.match(r'^[A-ZÄÖÜa-zäöüß\-]+,\s', q):           return "know-item"
    if re.search(r'\b(1[5-9]|20)\d{2}\b', q) and len(q.split()) >= 2:
        return "know-item"
    return None  # unklar → ML


# ── Hauptprogramm ──────────────────────────────────────────────────────────────

def gnd_lookup_fast(query: str, persons: dict, subjects: dict) -> dict:
    """
    Schneller GND-Lookup ohne Fuzzy-Matching.
    Nur exakte und Token-Treffer — O(1) pro Token, skaliert auf 100k+ Queries.
    """
    q = unicodedata.normalize("NFC", str(query)).lower().strip()
    tokens = [t for t in q.split() if len(t) > 2]

    result = {
        "person_exact": False, "subject_exact": False,
        "person_token": False, "subject_token": False,
        "conflict_same_token": False,
    }

    if q in persons: result["person_exact"] = True
    if q in subjects: result["subject_exact"] = True

    person_hit_token = subject_hit_token = None
    for token in tokens:
        if token in persons and not result["person_exact"] and not result["person_token"]:
            result["person_token"] = True
            person_hit_token = token
        if token in subjects and not result["subject_exact"] and not result["subject_token"]:
            result["subject_token"] = True
            subject_hit_token = token

    result["conflict_same_token"] = (
        person_hit_token is not None and
        subject_hit_token is not None and
        person_hit_token == subject_hit_token
    )
    return result


def classify_full(row: pd.Series) -> str | None:
    """Regelbasiert + GND. Gibt None zurück wenn ML entscheiden soll."""
    q = str(row.get("keyword", ""))
    n = len(q.split())

    # Strukturregeln
    rule = classify_rules(q)
    if rule:
        return rule

    # GND-basiert
    hat_p = row.get("person_exact") or row.get("person_token")
    hat_s = row.get("subject_exact") or row.get("subject_token")

    if hat_p and not hat_s: return "know-item"
    if hat_s and not hat_p: return "thematisch"
    if hat_p and hat_s:
        if row.get("conflict_same_token"): return "thematisch"
        return "know-item" if (row.get("person_exact") or row.get("person_token"))                else "thematisch"

    if re.match(r'^(der|die|das|ein|eine|the|a|an)\s', q, re.IGNORECASE):
        return "know-item" if n >= 4 else "thematisch"

    return None  # → ML


def main():
    print("📦 Lade Modell und GND-Index ...")
    with open(MODEL_FILE, "rb") as f:
        clf = pickle.load(f)
    with open(GND_FILE, "rb") as f:
        gnd_data = pickle.load(f)
    persons  = gnd_data["persons"]
    subjects = gnd_data["subjects"]
    sbert    = SentenceTransformer(SBERT_MODEL)
    print(f"   GND: {len(persons):,} Personen · {len(subjects):,} Sachbegriffe")

    print(f"\n📂 Lade {INPUT_FILE} ...")
    df = pd.read_csv(INPUT_FILE, sep=";", encoding="utf-8-sig", on_bad_lines="skip")
    df = df.dropna(subset=["keyword"])
    df["keyword"] = df["keyword"].str.strip()
    df = df[df["keyword"].str.len() > 1]
    print(f"   {len(df):,} Zeilen · {df['keyword'].nunique():,} unique Keywords")

    # GND-Lookup (schnell, kein Fuzzy)
    unique_kw = df["keyword"].drop_duplicates().tolist()
    print(f"\n🔍 GND-Lookup für {len(unique_kw):,} unique Keywords (kein Fuzzy) ...")
    lookup = {kw: gnd_lookup_fast(kw, persons, subjects) for kw in tqdm(unique_kw)}
    lookup_df = pd.DataFrame.from_dict(lookup, orient="index")
    lookup_df.index.name = "keyword"
    df = df.merge(lookup_df.reset_index(), on="keyword", how="left")

    # Regelbasiert + GND klassifizieren
    df["label"] = df.apply(classify_full, axis=1)

    # ML auf verbleibende None-Werte
    needs_ml = df["label"].isna()
    ml_queries = df[needs_ml]["keyword"].drop_duplicates().tolist()
    print(f"\n   Regelbasiert + GND entschieden: {(~needs_ml).sum():,}")
    print(f"   ML klassifiziert:                {needs_ml.sum():,}")

    if ml_queries:
        print("\n🤖 SBERT-Encoding ...")
        X = sbert.encode(ml_queries, show_progress_bar=True, batch_size=64)
        preds = dict(zip(ml_queries, clf.predict(X)))
        df.loc[needs_ml, "label"] = df.loc[needs_ml, "keyword"].map(preds)

    df.to_csv(OUTPUT_FILE, index=False, sep=";", encoding="utf-8-sig")

    print(f"\n✅ Gespeichert: {OUTPUT_FILE}")
    print("\n=== Klassenverteilung ===")
    print(df["label"].value_counts().to_string())
    ki = (df["label"] == "know-item").sum()
    th = (df["label"] == "thematisch").sum()
    if th > 0:
        print(f"\nVerhältnis KI/TH: {ki/th:.2f}  (Referenz: ~1.42)")


if __name__ == "__main__":
    main()